# 05 — Field profile validation: 1+1D vs 3+1D

Compares φ(z, t) directly between BubbleMaster (1+1D) and sledgehamr (3+1D)
at three snapshot times per validation point.

**Workflow**
1. Run **Section 1** to list available sledgehamr snapshot times; fill in `SNAP_INDICES`.
2. Run **Section 2** to write BubbleMaster setup files and SLURM scripts.
3. Submit: `! sbatch scripts/fields_<tag>.sh`
4. After jobs complete, run **Sections 3–5** to load and plot.

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import _cutoff_times, BubbleMasterParams

REPO_ROOT = Path(pt.__file__).parent.parent

PYSLEDGEHAMR_PATH = '/global/cfs/cdirs/m3166/buschman/sledgehamr'
sys.path.append(PYSLEDGEHAMR_PATH)
import pySledgehamr as sledgehamr

config = pt.Config()
params = BubbleMasterParams(dz=0.005, n_w=64, n_k=51,
                             how_often_ds=5, baby_steps=100, cutoff_type=0)

sh_base = '/pscratch/sd/b/buschman/other_runs/'

# Only the two axis-aligned (x) pairs — these are the only ones where a 1D
# slice from the 3D field passes directly through the collision axis.
#
# lb=0.84: bubble_N3_234_0_1_  → bubbles 0=(72.25,89.92,100) & 1=(127.8,89.92,100)
#           Δ=(55.55, 0, 0) — x-aligned, midpoint x=(72.25+127.8)/2=100.025
# lb=0.069: bubble_N3_456_0_1_ → bubbles 0=(16.23,19.06,22.50) & 1=(28.77,19.06,22.50)
#           Δ=(12.54, 0, 0) — x-aligned, midpoint x=(16.23+28.77)/2=22.50
VALIDATION_POINTS = [
    dict(lambda_bar=0.84, d=55.55,
         sh_path=sh_base+'bubble_N3_234_0_1_',
         sh_N0=2048, sh_L=200., sh_axis='x', sh_mid_x=100.025, sh_lev=0),
    dict(lambda_bar=0.069, d=12.54,
         sh_path=sh_base+'bubble_N3_456_0_1_',
         sh_N0=512,  sh_L=50.,  sh_axis='x', sh_mid_x=22.50,   sh_lev=0),
]

def gamma_for(lambda_bar, d):
    model = pt.PhysicsModel(pt.Phi4Potential(lambda_bar), config)
    return float(model.kinematics.Gamma(model.kinematics.collision_time(d)))

def vp_tag(vp):
    return f'lb{vp["lambda_bar"]}_g{vp["gamma"]:.4g}'

for vp in VALIDATION_POINTS:
    vp['gamma'] = gamma_for(vp['lambda_bar'], vp['d'])
    t_0, t_cut, t_m, _ = _cutoff_times(vp['d'], params.cutoff_type, params.t_0_scal)
    model   = pt.PhysicsModel(pt.Phi4Potential(vp['lambda_bar']), config)
    t_coll  = float(model.kinematics.collision_time(vp['d']))
    vp['t_coll'] = t_coll
    vp['t_m']    = t_m
    vp['t_mid']  = (t_coll + t_m) / 2.
    print(f'{vp_tag(vp)}  d={vp["d"]}  t_coll={t_coll:.3f}  t_mid={vp["t_mid"]:.3f}  t_m={t_m:.3f}')

## 1. List sledgehamr snapshot times

In [ ]:
# Lists available sledgehamr snapshot times and marks the three chosen ones.
# Chosen automatically: t_coll (collision), (t_coll+t_m)/2, t_m (cutoff midpoint).
for vp in VALIDATION_POINTS:
    output  = sledgehamr.Output(vp['sh_path'])
    t_snaps = output.GetTimesOfSlices()
    targets = [vp['t_coll'], vp['t_mid'], vp['t_m']]
    labels  = ['t_coll', 't_mid', 't_m']
    chosen  = {int(np.argmin(np.abs(t_snaps - t))): lbl for t, lbl in zip(targets, labels)}
    print(f"{vp_tag(vp)}  {Path(vp['sh_path']).name}")
    for i, t in enumerate(t_snaps):
        marker = f'  <-- {chosen[i]}' if i in chosen else ''
        print(f'  [{i:3d}]  t = {t:.4f}{marker}')
    print()

## 2. Prerequisites

Run notebook 04 first — it writes the SLURM scripts and submits the BubbleMaster jobs
(with `--save-fields`) that produce the `fields.h5` files loaded below.
The output lives in `validation/<tag>/out/`, shared with the spectrum comparison.

## 3. Helper functions

In [ ]:
def load_bm_fields(out_dir):
    """Return (s, z, phi) from fields.h5; phi has shape (n_s, n_z)."""
    with h5py.File(Path(out_dir) / 'fields.h5', 'r') as f:
        return f['s'][:], f['z'][:], f['phi'][:]


def extract_bm_band(s_bm, z_bm, phi_bm, t_snap):
    """
    For each z inside the light cone (z < t_snap), finds the two BM Milne-grid
    points bracketing s = sqrt(t_snap² - z²) and returns the field at both.
    Outside the light cone → NaN (BM only models the collision region).
    Also returns the bracketing Milne times at z=0 (the collision centre).
    """
    in_cone = z_bm < t_snap
    s_at_t  = np.where(in_cone, np.sqrt(np.maximum(t_snap**2 - z_bm**2, 0.)), 0.)
    i_lo    = np.clip(np.searchsorted(s_bm, s_at_t, side='right') - 1, 0, len(s_bm) - 2)
    iz      = np.arange(len(z_bm))
    phi_lo  = np.where(in_cone, phi_bm[i_lo,     iz], np.nan)
    phi_hi  = np.where(in_cone, phi_bm[i_lo + 1, iz], np.nan)

    # Milne bracket at z=0 (where s_at_t = t_snap)
    i_c    = np.clip(np.searchsorted(s_bm, t_snap, side='right') - 1, 0, len(s_bm) - 2)
    s_lo_c = s_bm[i_c]
    s_hi_c = s_bm[i_c + 1]
    return phi_lo, phi_hi, s_lo_c, s_hi_c


def get_sh_field_1d(output, snap_idx, L, mid_x, axis='x', lev=0):
    """
    1D profile along `axis` at the transverse midpoint row.
    Derives grid size from the actual returned array — no N0 needed.
    Returns (z_sh, phi_sh, t) with z_sh=0 at the collision midpoint.
    """
    slices  = output.GetSlice(snap_idx, axis, lev, ['Phi'])
    t       = slices['t']
    phi_2d  = slices['Phi']
    n_rows, n_cols = phi_2d.shape
    phi_1d  = phi_2d[n_rows // 2, :]
    z_sh    = np.linspace(0, L, n_cols) - mid_x
    return z_sh, phi_1d, t


def closest_snap(t_snaps, t_target):
    return int(np.argmin(np.abs(t_snaps - t_target)))

## 3. Load data

In [ ]:
# --- BubbleMaster fields ---
bm_data = {}
for vp in VALIDATION_POINTS:
    tag     = vp_tag(vp)
    model   = pt.PhysicsModel(pt.Phi4Potential(vp['lambda_bar']), config)
    out_dir = model.results_dir / 'validation' / tag / 'out'
    try:
        s, z, phi = load_bm_fields(out_dir)
        bm_data[tag] = dict(s=s, z=z, phi=phi)
        print(f'[{tag}]  s=[{s[0]:.3f},{s[-1]:.3f}]  z=[{z[0]:.3f},{z[-1]:.3f}]  phi shape={phi.shape}')
    except FileNotFoundError:
        print(f'[{tag}]  fields.h5 not found — run notebook 04 first')

# --- sledgehamr fields at three target times per VP ---
# Targets: collision time, midpoint, cutoff midpoint t_m.
# Some slices may have been purged; for each target we search outward from the
# closest index until we find one that can actually be read.
def load_snap_safe(output, t_snaps, t_target, label, vp):
    """Try indices in order of distance from t_target until one loads."""
    order = np.argsort(np.abs(t_snaps - t_target))
    for idx in order:
        try:
            z_sh, phi_sh, t_sh = get_sh_field_1d(
                output, int(idx), vp['sh_L'], vp['sh_mid_x'],
                axis=vp['sh_axis'], lev=vp['sh_lev'])
            if idx != order[0]:
                print(f'  {label}: preferred snap {order[0]} missing, used {idx} instead')
            return dict(label=label, idx=int(idx), t=t_sh, z=z_sh, phi=phi_sh)
        except Exception:
            continue
    return None

sh_data = {}
for vp in VALIDATION_POINTS:
    tag     = vp_tag(vp)
    output  = sledgehamr.Output(vp['sh_path'])
    t_snaps = output.GetTimesOfSlices()
    targets = [vp['t_coll'], vp['t_mid'], vp['t_m']]
    labels  = ['t_coll', 't_mid', 't_m']
    snaps   = []
    for t_target, lbl in zip(targets, labels):
        snap = load_snap_safe(output, t_snaps, t_target, lbl, vp)
        if snap is None:
            print(f'[{tag}]  {lbl}: no readable slice found — skipping VP')
            snaps = None
            break
        print(f'[{tag}]  {lbl}: snap {snap["idx"]}  t={snap["t"]:.4f}  (target {t_target:.4f})')
        snaps.append(snap)
    if snaps is not None:
        sh_data[tag] = snaps

## 4. Plot

In [ ]:
n_vps   = len(VALIDATION_POINTS)
n_snaps = 3
colors  = ['#0072B2', '#D55E00']

fig, axes = plt.subplots(n_snaps, n_vps,
                          figsize=(5 * n_vps, 3.5 * n_snaps),
                          squeeze=False)

for col, (vp, color) in enumerate(zip(VALIDATION_POINTS, colors)):
    tag = vp_tag(vp)
    d   = vp['d']
    bm  = bm_data.get(tag)
    sh  = sh_data.get(tag, [])

    axes[0, col].set_title(
        rf"$\bar{{\lambda}}$={vp['lambda_bar']}, $\gamma$={vp['gamma']:.3g}",
        fontsize=9)

    for row, snap in enumerate(sh):
        ax     = axes[row, col]
        t_snap = snap['t']

        # 3+1D line
        mask = snap['z'] >= 0
        ax.plot(snap['z'][mask], snap['phi'][mask],
                color=color, lw=2, label='3+1D')

        # 1+1D band
        bm_label = '1+1D'
        if bm is not None:
            if t_snap > bm['s'][-1]:
                ax.text(0.05, 0.05, f't={t_snap:.2f} > smax={bm["s"][-1]:.2f}',
                        transform=ax.transAxes, fontsize=7, color='red', va='bottom')
            else:
                phi_lo, phi_hi, s_lo, s_hi = extract_bm_band(
                    bm['s'], bm['z'], bm['phi'], t_snap)
                ax.fill_between(bm['z'], phi_lo, phi_hi,
                                color=color, alpha=0.35, linewidth=1, label='1+1D')
                bm_label = f'1+1D  s=[{s_lo:.3f}, {s_hi:.3f}]'

        ax.text(0.97, 0.95,
                f"3+1D: t = {t_snap:.3f} ({snap['label']})\n{bm_label}",
                transform=ax.transAxes, ha='right', va='top', fontsize=7)
        ax.set_xlim(0, d * 0.7)
        if col == 0:
            ax.set_ylabel(r'$\phi$')
        if row == n_snaps - 1:
            ax.set_xlabel('z')
        if row == 0 and col == 0:
            ax.legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'validate_fields.pdf', bbox_inches='tight')
plt.show()